In [6]:
!pip install langchain==0.3.27
!pip install openai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-community 0.4.2 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langchain-experimental 0.3.4 requires langchain-community<0.4.0,>=0.3.0, but you have langchain-community 0.4.2 which is incompatible.
langchain-google-genai 4.4.0 requires langchain-core<2.0.0,>=1.6.1, but you have langchain-core 0.3.86 which is incompatible.
langchain-openai 1.7.1 requires langchain-core<2.0.0,>=1.6.10, but you have langchain-core 0.3.86 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.

  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
Using cached langchain_text_splitters-0.3.11-py3-none-any.whl (33 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.10
    Uninstalling langchain-core-1.6.10:
      Successfully uninstalled langchain-core-1.6.10
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.3
    Uninstalling langchain-text-splitters-1.1.3:
      Successfully uninstalled langchain-text-splitters-1.1.3



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma, FAISS
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import JSONLoader, CSVLoader
from langchain.document_loaders import DirectoryLoader
import os

In [18]:
csv_loader = DirectoryLoader("../data", glob="*.csv", loader_cls=CSVLoader, loader_kwargs={"csv_args": {"delimiter": ","}})


In [19]:
medical_docs = csv_loader.load()

In [18]:
medical_docs[0].page_content

"Unnamed: 0: 0\ndisease: Atrial fibrillation\nmain_link: https://www.mayoclinic.org/diseases-conditions/atrial-fibrillation/symptoms-causes/syc-20350624\nDiagnosis_treatment_link: https://www.mayoclinic.org/diseases-conditions/atrial-fibrillation/diagnosis-treatment/drc-20350630\nDoctors_departments_link: https://www.mayoclinic.org/diseases-conditions/atrial-fibrillation/doctors-departments/ddc-20350632\nOverview: Atrial fibrillation (AFib) is an irregular and often very rapid heart rhythm. An irregular heart rhythm is called an arrhythmia.AFibcan lead to blood clots in the heart. The condition also increases the risk of stroke, heart failure and other heart-related complications.\r\nDuring atrial fibrillation, the heart's upper chambers â€” called the atria â€” beat chaotically and irregularly. They beat out of sync with the lower heart chambers, called the ventricles. For many people,AFibmay have no symptoms. ButAFibmay cause a fast, pounding heartbeat, shortness of breath or light-h

In [19]:
len(medical_docs)

2887

In [10]:
azure_embeddings_api_key = os.getenv("azure_embeddings_api_key")
azure_embeddings_endpoint = os.getenv("azure_embeddings_endpoint")
azure_embeddings_deployment = os.getenv("azure_embeddings_deployment")
azure_api_version = os.getenv("OPENAI_API_VERSION")
#2 Load The DataSet

In [11]:
from openai import AzureOpenAI

In [43]:
%pip install -U langchain-openai


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from openai import AzureOpenAI

In [1]:
from langchain_openai import AzureOpenAIEmbeddings

In [14]:
from langchain_openai import AzureOpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

# Create a NEW LangChain embedding object
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.getenv("AZURE_OPENAI_EMBEDDING_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_EMBEDDING_API_KEY"),
    openai_api_version=os.getenv("AZURE_OPENAI_EMBEDDING_VERSION"),
    azure_deployment="text-embedding-ada-002"
)

print(type(embeddings))
print(hasattr(embeddings, "embed_documents"))
print(hasattr(embeddings, "embed_query"))

<class 'langchain_openai.embeddings.azure.AzureOpenAIEmbeddings'>
True
True


In [20]:
medical_vectorstore = FAISS.from_documents(  documents=medical_docs,
    embedding=embeddings)

RateLimitError: Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Your requests to text-embedding-ada-002 for text-embedding-ada-002 in East US have exceeded the call rate limit for your current AIServices S0 pricing tier. This request was for Embeddings_Create under Azure OpenAI API version 2024-10-21. Please retry after 60 seconds. To increase your default rate limit, visit: https://aka.ms/oai/quotaincrease.'}}

In [41]:
print("Number of documents:", len(medical_docs))
print("First document type:", type(medical_docs[0]))
print("First document:", medical_docs[0])

Number of documents: 2887
First document type: <class 'langchain_core.documents.base.Document'>
First document: page_content='Unnamed: 0: 0
disease: Atrial fibrillation
main_link: https://www.mayoclinic.org/diseases-conditions/atrial-fibrillation/symptoms-causes/syc-20350624
Diagnosis_treatment_link: https://www.mayoclinic.org/diseases-conditions/atrial-fibrillation/diagnosis-treatment/drc-20350630
Doctors_departments_link: https://www.mayoclinic.org/diseases-conditions/atrial-fibrillation/doctors-departments/ddc-20350632
Overview: Atrial fibrillation (AFib) is an irregular and often very rapid heart rhythm. An irregular heart rhythm is called an arrhythmia.AFibcan lead to blood clots in the heart. The condition also increases the risk of stroke, heart failure and other heart-related complications.
During atrial fibrillation, the heart's upper chambers â€” called the atria â€” beat chaotically and irregularly. They beat out of sync with the lower heart chambers, called the ventricles. 

In [27]:
print("Embedding object type:", type(openai_embeddings))
print("Has embed_documents:", hasattr(openai_embeddings, "embed_documents"))
print("Has embed_query:", hasattr(openai_embeddings, "embed_query"))

Embedding object type: <class 'openai.lib.azure.AzureOpenAI'>
Has embed_documents: False
Has embed_query: False


In [3]:
from openai import OpenAI

In [4]:
from langchain_openai import AzureChatOpenAI

In [25]:
from langchain_openai import ChatOpenAI

In [21]:
llm = AzureChatOpenAI(
    api_key=os.getenv("AZURE_OPENAI_CHAT_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_CHAT_ENDPOINT"),
    api_version="2025-01-01-preview"
)

In [26]:
llm = ChatOpenAI(
    base_url=os.getenv("AZURE_OPENAI_CHAT_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_CHAT_API_KEY"),
    model="gpt-5-mini"
)

In [27]:
prompt = "Tell Me A Joke On Artificiall Intelligence"
responce = llm.invoke(prompt)

In [28]:
print(responce)

content='Sure — here are a few AI jokes:\n\n- Why did the AI go to therapy? It had too many unresolved layers.  \n- I asked my AI to tell a joke about AI. It replied, “Processing…” for 18 hours — turns out it was training on stand‑up routines.  \n- What’s an AI’s favorite dance? The algorithm.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 402, 'prompt_tokens': 14, 'total_tokens': 416, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 31, 'engine_ttlt_ms': 3423, 'pre_inference_ms': 107, 'service_tbt_ms': 8, 'service_ttft_ms': 444, 'service_ttlt_ms': 3830, 'user_visible_ttft_ms': 337}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EYQTgVmDhTbRzqnhscIGN5ZA9QcPc', 'service_tier': 'default', '

In [7]:
def start_chat():
    print("Azure AI Foundry Client (type 'exit' or 'quit' to terminate)")
    messages = [
        {"role": "system", "content": "You are a friendly AI Assistant."}
    ]
    while True:
        user_input = input("User: ")
        if user_input.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break
        messages.append({"role": "user", "content": user_input})
        response = llm.chat.completions.create(
            model=azure_deployment,
            messages=messages
        )
        response = response.choices[0].message.content
        print(f"Bot: {response}")
        # update message with previous conversation
        messages.append({"role": "assistant", "content": response})
 
 
 

In [9]:
start_chat()

Azure AI Foundry Client (type 'exit' or 'quit' to terminate)


AttributeError: 'AzureChatOpenAI' object has no attribute 'chat'